# Step 11 · Finale: SFT vs RL vs RLHF

Same questions, same settings, side by side.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
from lab import engines
models = [m for m in (BASE, latest("sft"), latest("rl"), latest("rlhf")) if m]
qs = [t["prompt"] for t in data.active_test_rows(data.load(latest_ds("test")))][:4]
table = {"question": qs}
for mid in models:
    table[mid] = engines.get(registry.model(mid)["base_model"]).generate(mid, qs, max_new_tokens=80)
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(table)

In [ ]:
rules = registry.rewards()[0]["rules"]
pd.Series({mid: sum(rewards.score(a, rules)["total"] for a in table[mid]) / len(qs) for mid in models}, name="mean rule score")